In [ ]:
!pip install -r requirements.txt

In [1]:
# Librairies utilisées
import re
import time
import warnings
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

warnings.filterwarnings("ignore")


In [26]:
df_naf84 = pd.read_csv(r"C:\Users\zargo\Documents\ENSAE 3A\Economics of Energy Markets\consommation-annuelle-d-electricite-et-gaz-par-commune-naf-84.csv", sep=";")
df_total = pd.read_csv(r"C:\Users\zargo\Documents\ENSAE 3A\Economics of Energy Markets\consommation-annuelle-d-electricite-et-gaz-par-commune.csv", sep=";")

In [27]:
df_total["Code Commune"].nunique(), df_naf84["Code Commune"].nunique()

(67291, 15231)

In [15]:
df_total_elec = df_total[df_total['FILIERE']=='Electricité'].groupby(["Code Commune", "Année"], as_index=False).agg(
        conso_totale_mwh     = ("Conso totale (MWh)",     "sum"),
    )

In [21]:
df_naf84_elec = df_naf84[df_naf84['FILIERE']=='Electricité'].groupby(["Code Commune", "Année"], as_index=False).agg(
        conso_naf84_mwh     = ("Conso totale (MWh)",     "sum"),
    )

In [6]:
def fetch_cerema_data(file_path="cerema_data.parquet"):
    """
    Cerema satellite-based detection of public-lighting changes.
    Columns expected:
      code_commune, annee_detection, type_changement
      (total_extinction | extinction_partielle | abandon_extinction | extension)
    """

    gdf = gpd.read_file(file_path, engine="pyogrio")

    return gdf

df_cerema_raw = fetch_cerema_data(r"C:\Users\zargo\Documents\ENSAE 3A\Economics of Energy Markets\vectorextinctionfrance\vectorExtinctionFrance.gpkg")
if df_cerema_raw is not None:
    print("Columns:", df_cerema_raw.columns.tolist())

Columns: ['insee_com', 'nom', 'insee_dep', 'insee_reg', 'Date Extinction EP', 'Date Renov parc / extinction', "Date Abandon d'extinction", 'Date Extension EP', 'changes_EP', '2014-01', '2014-02', '2014-03', '2014-04', '2014-05', '2014-06', '2014-07', '2014-08', '2014-09', '2014-10', '2014-11', '2014-12', '2015-01', '2015-02', '2015-03', '2015-04', '2015-05', '2015-06', '2015-07', '2015-08', '2015-09', '2015-10', '2015-11', '2015-12', '2016-01', '2016-02', '2016-03', '2016-04', '2016-05', '2016-06', '2016-07', '2016-08', '2016-09', '2016-10', '2016-11', '2016-12', '2017-01', '2017-02', '2017-03', '2017-04', '2017-05', '2017-06', '2017-07', '2017-08', '2017-09', '2017-10', '2017-11', '2017-12', '2018-01', '2018-02', '2018-03', '2018-04', '2018-05', '2018-06', '2018-07', '2018-08', '2018-09', '2018-10', '2018-11', '2018-12', '2019-01', '2019-02', '2019-03', '2019-04', '2019-05', '2019-06', '2019-07', '2019-08', '2019-09', '2019-10', '2019-11', '2019-12', '2020-01', '2020-02', '2020-03', '

In [7]:
df_cerema_raw["extinction_totale"] = df_cerema_raw["changes_EP"].str.contains("E", na=False).astype(int)
df_cerema_raw["renovation"]        = df_cerema_raw["changes_EP"].str.contains("R", na=False).astype(int)
df_cerema_raw["abandon"]           = df_cerema_raw["changes_EP"].str.contains("D", na=False).astype(int)
df_cerema_raw["extension"]         = df_cerema_raw["changes_EP"].str.contains("A", na=False).astype(int)

# Recherche de la première année d'extinction
def extract_first_year(s):
    """Extract the year from a string like ['2022-11'] or '2022-11'."""
    if pd.isna(s) or str(s).strip() in ("", "nan", "None"):
        return np.nan
    years_found = re.findall(r"(\d{4})-\d{2}", str(s))
    return int(years_found[0]) if years_found else np.nan

df_cerema_raw["annee_extinction"]         = df_cerema_raw["Date Extinction EP"].apply(extract_first_year)
df_cerema_raw["annee_renovation"]         = df_cerema_raw["Date Renov parc / extinction"].apply(extract_first_year)
df_cerema_raw["annee_abandon_extinction"] = df_cerema_raw["Date Abandon d'extinction"].apply(extract_first_year)
df_cerema_raw["annee_extension"]          = df_cerema_raw["Date Extension EP"].apply(extract_first_year)

# Récupération de la lumniosité
monthly_cols = sorted([c for c in df_cerema_raw.columns if re.match(r"^\d{4}-\d{2}$", c)])

# Statistique au niveau des communes
df_cerema_raw["luminosite_totale"]   = df_cerema_raw[monthly_cols].mean(axis=1, skipna=True)
df_cerema_raw["nb_mois_luminosite"]  = df_cerema_raw[monthly_cols].notna().sum(axis=1)

# luminosité sur l'année spécifiée
radiance_long_parts = []
for yr in YEARS:
    yr_cols = [c for c in monthly_cols if c.startswith(str(yr))]
    if not yr_cols:
        continue
    tmp = df_cerema_raw[["insee_com"] + yr_cols].copy()
    tmp["annee"] = yr
    tmp["luminosite_annuelle"] = tmp[yr_cols].mean(axis=1, skipna=True)
    radiance_long_parts.append(tmp[["insee_com", "annee", "luminosite_annuelle"]])

radiance_by_year = pd.concat(radiance_long_parts, ignore_index=True)

# Construction de la table CEREMA
STATIC_CEREMA_COLS = [
    "insee_com",
    "extinction_totale", "renovation", "abandon", "extension",
    "annee_extinction", "annee_renovation", "annee_abandon_extinction", "annee_extension",
    "luminosite_totale", "nb_mois_luminosite",
]
cerema_static = df_cerema_raw[STATIC_CEREMA_COLS].copy()
cerema_static["insee_com"] = cerema_static["insee_com"].astype(str).str.zfill(5)

# Merge sur l'année spécifique retenue
radiance_by_year["insee_com"] = radiance_by_year["insee_com"].astype(str).str.zfill(5)

print(f"  CEREMA static : {len(cerema_static):,} communes")
print(f"  Radiance long : {len(radiance_by_year):,} rows (commune × year)")
print(f"  extinction_totale: {cerema_static['extinction_totale'].sum():,} communes")
print(f"  renovation       : {cerema_static['renovation'].sum():,} communes")
print(f"  abandon          : {cerema_static['abandon'].sum():,} communes")
print(f"  extension        : {cerema_static['extension'].sum():,} communes")
print()



  CEREMA static : 34,813 communes
  Radiance long : 243,691 rows (commune × year)
  extinction_totale: 11,978 communes
  renovation       : 3,545 communes
  abandon          : 510 communes
  extension        : 131 communes



In [31]:
radiance_by_year

,insee_com,annee,luminosite_annuelle
0,01001,2018,2.683942
1,01002,2018,0.649227
2,01004,2018,15.257463
3,01005,2018,0.822826
4,01006,2018,0.578600
...,...,...,...
243686,95676,2024,1.385574
243687,95678,2024,1.321057
243688,95680,2024,26.734741
243689,95682,2024,1.391425


In [35]:
df_total_elec['Code Commune'] = df_total_elec['Code Commune'].astype(str).str.zfill(5)

In [36]:
df_naf84_elec

,Code Commune,Année,conso_naf84_mwh
0,01001,2018,211.313500
1,01001,2019,211.392007
2,01001,2020,21.579998
3,01001,2021,15.521903
4,01001,2022,28.299000
...,...,...,...
95158,97610,2023,2372.211000
95159,97611,2023,2002.506000
95160,97614,2023,37.303000
95161,97615,2023,669.272000


In [42]:
# Construction du panel: NAF84 + total + CEREMA
print("=" * 65)
print("PANEL CONSTRUCTION")
print("=" * 65)

panel = (
    df_naf84_elec
    .merge(
        df_total_elec[["Code Commune", "Année", "conso_totale_mwh"]],
        on=["Code Commune", "Année"],
        how="left",
    )

    .merge(
        cerema_static,
        left_on="Code Commune",
        right_on="insee_com",
        how="left",
    )
    .drop(columns="insee_com", errors="ignore")

    .merge(
        radiance_by_year,
        left_on=["Code Commune", "Année"],
        right_on=["insee_com", "annee"],
        how="left",
    )
    .drop(columns="insee_com", errors="ignore")
)

# Variables dérivées
panel["naf84_share"] = panel["conso_naf84_mwh"] / panel["conso_totale_mwh"].clip(lower=1)
panel["ln_naf84"]    = np.log(panel["conso_naf84_mwh"].clip(lower=1))
panel["ln_total"]    = np.log(panel["conso_totale_mwh"].clip(lower=1))

for col in ["conso_naf84_mwh", "conso_totale_mwh", "naf84_share"]:
    lo, hi = panel[col].quantile([0.01, 0.99])
    panel[col] = panel[col].clip(lo, hi)

# Verifications
print(f"Panel: {panel['Code Commune'].nunique():,} communes × {panel['Année'].nunique()} years = {len(panel):,} obs")
print(f"  naf84_share  — mean: {panel['naf84_share'].mean():.3f}  "
      f"median: {panel['naf84_share'].median():.3f}  max: {panel['naf84_share'].max():.3f}")
print(f"  luminosite   — mean: {panel['luminosite_annuelle'].mean():.3f}  "
      f"non-null: {panel['luminosite_annuelle'].notna().sum():,}")
print(f"  CEREMA merge — extinction non-null: {panel['annee_extinction'].notna().sum():,}")

assert panel["naf84_share"].max() <= 1.01, "naf84_share > 1 — double-counting?"
assert panel["naf84_share"].mean() < 0.40, "naf84_share mean suspiciously high"
print("ok")

# Enregistrement sous format parquet
panel.to_parquet(   "C:/Users/zargo/Documents/ENSAE 3A/Economics of Energy Markets/panel.parquet",      index=False)

print(f"  panel.parquet     — {len(panel):,} rows")
print()
print("Panel columns:")
print(panel.dtypes.to_string())

PANEL CONSTRUCTION
Panel: 15,101 communes × 7 years = 95,227 obs
  naf84_share  — mean: 0.027  median: 0.015  max: 0.272
  luminosite   — mean: 4.495  non-null: 94,623
  CEREMA merge — extinction non-null: 48,648
ok
  panel.parquet     — 95,227 rows

Panel columns:
Code Commune                    str
Année                         int64
conso_naf84_mwh             float64
conso_totale_mwh            float64
extinction_totale           float64
renovation                  float64
abandon                     float64
extension                   float64
annee_extinction            float64
annee_renovation            float64
annee_abandon_extinction    float64
annee_extension             float64
luminosite_totale           float64
nb_mois_luminosite          float64
annee                       float64
luminosite_annuelle         float64
naf84_share                 float64
ln_naf84                    float64
ln_total                    float64


In [43]:
panel

,Code Commune,Année,conso_naf84_mwh,conso_totale_mwh,extinction_totale,renovation,abandon,extension,annee_extinction,annee_renovation,annee_abandon_extinction,annee_extension,luminosite_totale,nb_mois_luminosite,annee,luminosite_annuelle,naf84_share,ln_naf84,ln_total
0,01001,2018,211.313500,3245.912185,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,2.461964,132.0,2018.0,2.683942,0.065101,5.353343,8.085152
1,01001,2019,211.392007,3244.387981,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,2.461964,132.0,2019.0,2.590463,0.065156,5.353714,8.084682
2,01001,2020,21.579998,3351.849802,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,2.461964,132.0,2020.0,2.290969,0.006438,3.071767,8.117268
3,01001,2021,15.521903,3597.380676,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,2.461964,132.0,2021.0,2.255160,0.004315,2.742252,8.187961
4,01001,2022,28.299000,3433.786000,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,2.461964,132.0,2022.0,2.378538,0.008241,3.342826,8.141419
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95222,97610,2023,2372.211000,56008.100200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.042355,7.771578,10.933252
95223,97611,2023,2002.506000,117023.841700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.017112,7.602155,11.670133
95224,97614,2023,37.303000,9645.394100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.003867,3.619074,9.174236
95225,97615,2023,669.272000,26848.428600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.024928,6.506191,10.197963
